## Реализация нейронной сети в Python

Ваша задача реализовать, используя только библиотеку NumPy, класс, соответствующий многослойной нейронной сети, а также все вспомогательные средства для обучения и инференса сети.

### Минимальные требования

1. Класс должен иметь метод `predict`, который по данному входу вычисляет предсказание.
2. Класс должен иметь возможность задавать количество слоёв и количество нейронов в каждом слое, а также функции активации каждого слоя. В качестве функций активации добавьте поддержку: тождественной, ReLU, LeakyReLU, tanh и sigmoid.
3. Сделайте возможность инициализировать веса как хардкодом, так и каким-то методом инициализации, например Xavier.
4. Класс должен иметь режим обучения с функцией потерь MSE (обратите внимание, что на семинарах мы разбирали не совсем MSE). Как вы его реализуете, решайте сами. Это может быть отдельный метод класса или какой-то флаг, как в PyTorch.
5. Обучение происходит с помощью обычного градиентного спуска с параметром $\alpha$ (learning rate).
6. На каком-либо примере (можно искусственном) показать примеры, где ваша модель учится предсказывать. Соответственно приложить графики лосса, сравнения предсказаний и реальных откликов и т.д.
7. Поддерживать хороший стиль кода ([PEP 8](https://peps.python.org/pep-0008/)). К классу и методам нужно написать докстринги, чтобы можно было быстро разобраться, что умеет ваш класс.

Нельзя пользоваться LLM, если только не в целях нахождения какого-нибудь труднонаходимого бага. Если у вас возникает желание скормить это всё в нейронку, чтобы она за вас всё сделала, то лучше вообще не делайте. Эта работа прежде всего нужна для того, чтобы полностью разобраться в основах работы МНС и метода обратного распространения. Делиться кодом тоже нельзя, иначе не буду засчитывать обоим.

### В дополнении

Вы можете попробовать сделать что-то больше плана минимум. Например, поддержку более продвинутых оптимизационных алгоритмов, поддержку других лосс-функций и т.д.. Тут всё зависит от вашего полёта фантазии и желания.

Всякие дополнительные библиотеки, которые не связаны с нейронными сетями, но помогают делать код лучше/легче (Pydantic, например) приветствуются.

### Удачи!

In [ ]:
# YOUR CODE HEREimport numpy as np
import matplotlib.pyplot as plt
import numpy as np

from layer.layer import DenseLayer
from neural_network import *
from loss_functions import *
from initialization import *
from layer import *
from activation_functions import tanh
from neural_network.seq_nn import SequentialNeuralNetwork

np.random.seed(42)

x_train = np.linspace(-np.pi, np.pi, 256).reshape(-1, 1)
y_train = np.sin(x_train)

network = SequentialNeuralNetwork(
    modules=[
        DenseLayer(1, 32),
        Tanh(),
        DenseLayer(32, 32),
        Tanh(),
        DenseLayer(32, 1)
    ],
    loss=MSELoss()
)

history = network.fit(
    inputs=x_train,
    target=y_train,
    epochs=6000,
    learning_rate=0.01
)

x_test = np.linspace(-np.pi, np.pi, 400).reshape(-1, 1)
y_true = np.sin(x_test)
y_pred = network.predict(x_test)

control_x = np.array([
    [-np.pi],
    [-np.pi / 2],
    [0.0],
    [np.pi / 2],
    [np.pi]
])
control_true = np.sin(control_x)
control_pred = network.predict(control_x)

plt.figure(figsize=(8, 5))
plt.plot(history)
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("mse loss")
plt.title("loss during training")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(x_test.ravel(), y_true.ravel(), label="sin(x)")
plt.plot(x_test.ravel(), y_pred.ravel(), label="prediction")
plt.xlabel("x")
plt.ylabel("y")
plt.title("sin(x) approximation")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(control_x.ravel(), control_true.ravel(), label="true")
plt.scatter(control_x.ravel(), control_pred.ravel(), label="pred")
plt.xlabel("x")
plt.ylabel("y")
plt.title("control points")
plt.legend()
plt.grid(True)
plt.show()

print("final loss:", history[-1])
print()

for x_value, true_value, pred_value in zip(control_x.ravel(), control_true.ravel(), control_pred.ravel()):
    print(
        f"x={x_value: .4f} true={true_value: .6f} pred={pred_value: .6f} abs_err={abs(true_value - pred_value): .6f}"
    )